# EdgeFace-XS Evaluation

Evaluates fine-tuned EdgeFace-XS model against stock pretrained model.

**Two evaluation modes:**
1. **Classification accuracy** on val split of `train.csv` (sanity check matching fine-tune setup)
2. **Embedding quality**: cosine similarity between same-class pairs vs different-class pairs

> ⚠️ **Note:** Current dataset has class labels like `YOUNG`/`MIDDLE`/`OLD` (age groups), not face identities. For true face verification (NHAI worker attendance use case), an identity-labeled dataset (IndicFairFace / IMFDB) is required. This notebook still gives useful signal on embedding quality.

**If DFW dataset is available** at `DFW_ROOT`, also evaluates TAR@FAR=1e-4 per disguise type — see optional last section.

In [ ]:
!pip install -q torch torchvision timm scikit-learn pillow pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = '/content/drive/MyDrive/nhai_face/dataset'
CHECKPOINT_DIR = '/content/drive/MyDrive/nhai_face/checkpoints'

FINETUNED_PATH = f'{CHECKPOINT_DIR}/edgeface_xs_indian_best.pt'
STOCK_PATH = '/content/drive/MyDrive/nhai_face/edgeface_xs_gamma_06.pt'
DFW_ROOT = '/content/drive/MyDrive/nhai_face/dfw'  # optional: DFW dataset root

import os
print(f'Fine-tuned exists: {os.path.exists(FINETUNED_PATH)}')
print(f'Stock exists:      {os.path.exists(STOCK_PATH)}')
print(f'DFW dataset exists: {os.path.exists(DFW_ROOT)}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torchvision import transforms
from sklearn.metrics import roc_curve, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# --- Model architecture (same as finetune notebook) ---
class LoRaLin(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=True):
        super().__init__()
        self.linear1 = nn.Linear(in_features, rank, bias=False)
        self.linear2 = nn.Linear(rank, out_features, bias=bias)
    def forward(self, x):
        return self.linear2(self.linear1(x))


def replace_linear_with_lowrank(model, rank_ratio=0.2):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear) and 'head' not in name:
            in_f, out_f = module.in_features, module.out_features
            rank = max(2, int(min(in_f, out_f) * rank_ratio))
            setattr(model, name, LoRaLin(in_f, out_f, rank, bias=module.bias is not None))
        else:
            replace_linear_with_lowrank(module, rank_ratio)
    return model


def build_edgeface_xs():
    model = timm.create_model('edgenext_x_small')
    model.reset_classifier(512)
    model = replace_linear_with_lowrank(model, rank_ratio=0.6)
    return model

In [ ]:
# --- Load both models (handles all checkpoint formats) ---

def load_model(weights_path):
    model = build_edgeface_xs()
    ckpt = torch.load(weights_path, map_location='cpu')

    # Case 1: dict wrapper (e.g., _last.pt with {'backbone': ..., 'optimizer': ...})
    if isinstance(ckpt, dict) and 'backbone' in ckpt:
        print(f'  Detected wrapped checkpoint (epoch {ckpt.get("epoch", "?")}, val_acc {ckpt.get("val_acc", "?")})')
        state_dict = ckpt['backbone']
    # Case 2: plain state_dict (most common — _best.pt, stock)
    elif isinstance(ckpt, dict):
        state_dict = ckpt
    else:
        raise ValueError(f'Unexpected checkpoint type: {type(ckpt)}')

    # Strip "model." prefix (stock checkpoint has it, fine-tuned doesn't)
    new_state_dict = {k.replace('model.', '', 1) if k.startswith('model.') else k: v
                      for k, v in state_dict.items()}

    missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
    print(f'  Missing: {len(missing)}, Unexpected: {len(unexpected)}')
    if missing:    print(f'    Missing sample: {missing[:3]}')
    if unexpected: print(f'    Unexpected sample: {unexpected[:3]}')
    model.eval().to(device)
    return model

print('Loading fine-tuned model...')
model_ft = load_model(FINETUNED_PATH)

print('\nLoading stock model...')
model_stock = load_model(STOCK_PATH)

print('\n✅ Both models loaded.')

In [ ]:
# --- Build val split from train.csv (same as finetune notebook) ---

preprocess = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(112),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

@torch.no_grad()
def get_embedding(model, img_path):
    img = Image.open(img_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    emb = model(tensor)
    return F.normalize(emb, dim=1).cpu().numpy().flatten()

@torch.no_grad()
def get_embeddings_batch(model, img_paths, batch_size=64):
    """Faster batch embedding extraction."""
    embeddings = []
    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i:i+batch_size]
        tensors = []
        for p in batch_paths:
            img = Image.open(p).convert('RGB')
            tensors.append(preprocess(img))
        batch = torch.stack(tensors).to(device)
        emb = model(batch)
        emb = F.normalize(emb, dim=1).cpu().numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

# Recreate the same val split as fine-tuning (random_state=42)
full_df = pd.read_csv(os.path.join(DATASET_ROOT, 'train.csv'))
label_col = full_df.columns[-1]
img_col = full_df.columns[0]

counts = full_df[label_col].value_counts()
valid_classes = counts[counts >= 2].index
filtered_df = full_df[full_df[label_col].isin(valid_classes)].reset_index(drop=True)

_, val_df = train_test_split(
    filtered_df, test_size=0.1, stratify=filtered_df[label_col], random_state=42,
)
val_df = val_df.reset_index(drop=True)

train_img_dir = os.path.join(DATASET_ROOT, 'train')
val_img_paths = [os.path.join(train_img_dir, str(f)) for f in val_df[img_col]]
val_labels = val_df[label_col].astype(str).tolist()

print(f'Val set: {len(val_df)} images, {len(set(val_labels))} classes')
print(f'Class distribution: {val_df[label_col].value_counts().to_dict()}')

In [ ]:
# --- Eval 1: Embedding-based pair verification (same-class vs different-class) ---
# Build positive pairs (same class) and negative pairs (different class).
# Since labels are age groups (not identity), this measures whether embeddings
# cluster meaningfully by class — useful as a coarse embedding-quality signal.

import random
random.seed(42)

def build_pairs(labels, n_pos=2000, n_neg=2000):
    label_to_idx = defaultdict(list)
    for i, lab in enumerate(labels):
        label_to_idx[lab].append(i)

    pos_pairs, neg_pairs = [], []
    # Positive: same class
    classes = list(label_to_idx.keys())
    for _ in range(n_pos):
        c = random.choice(classes)
        if len(label_to_idx[c]) < 2:
            continue
        i, j = random.sample(label_to_idx[c], 2)
        pos_pairs.append((i, j))
    # Negative: different class
    for _ in range(n_neg):
        c1, c2 = random.sample(classes, 2)
        i = random.choice(label_to_idx[c1])
        j = random.choice(label_to_idx[c2])
        neg_pairs.append((i, j))
    return pos_pairs, neg_pairs


def evaluate_embedding_quality(model, name):
    print(f'Computing embeddings for {name}...')
    emb = get_embeddings_batch(model, val_img_paths, batch_size=64)
    print(f'  Embeddings shape: {emb.shape}')

    pos_pairs, neg_pairs = build_pairs(val_labels)

    pos_scores = np.array([float(emb[i] @ emb[j]) for i, j in pos_pairs])
    neg_scores = np.array([float(emb[i] @ emb[j]) for i, j in neg_pairs])

    scores = np.concatenate([pos_scores, neg_scores])
    labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    fpr, tpr, _ = roc_curve(labels, scores)

    # TAR at various FAR levels
    tar_at = {}
    for target_far in [1e-1, 1e-2, 1e-3]:
        idx = np.searchsorted(fpr, target_far)
        tar_at[target_far] = tpr[min(idx, len(tpr)-1)]

    print(f'  Pos mean: {pos_scores.mean():.4f} | Neg mean: {neg_scores.mean():.4f}')
    print(f'  Gap (higher = better): {pos_scores.mean() - neg_scores.mean():.4f}')
    print(f'  TAR@FAR=1e-1: {tar_at[1e-1]:.4f}')
    print(f'  TAR@FAR=1e-2: {tar_at[1e-2]:.4f}')
    print(f'  TAR@FAR=1e-3: {tar_at[1e-3]:.4f}')
    return {
        'pos_mean': pos_scores.mean(),
        'neg_mean': neg_scores.mean(),
        'gap': pos_scores.mean() - neg_scores.mean(),
        'tar_far_1e1': tar_at[1e-1],
        'tar_far_1e2': tar_at[1e-2],
        'tar_far_1e3': tar_at[1e-3],
        'embeddings': emb,
    }

print('=== Fine-tuned model ===')
ft_results = evaluate_embedding_quality(model_ft, 'fine-tuned')

print('\n=== Stock model ===')
stock_results = evaluate_embedding_quality(model_stock, 'stock')

In [ ]:
# --- Summary comparison ---
print('=' * 70)
print(f'{"Metric":<25} {"Stock":>15} {"Fine-tuned":>15} {"Delta":>10}')
print('-' * 70)
metrics = [
    ('Pos pair similarity',   'pos_mean'),
    ('Neg pair similarity',   'neg_mean'),
    ('Pos-Neg gap',           'gap'),
    ('TAR @ FAR=1e-1',        'tar_far_1e1'),
    ('TAR @ FAR=1e-2',        'tar_far_1e2'),
    ('TAR @ FAR=1e-3',        'tar_far_1e3'),
]
for label, key in metrics:
    s = stock_results[key]
    f = ft_results[key]
    delta = f - s
    sign = '+' if delta >= 0 else ''
    print(f'{label:<25} {s:>15.4f} {f:>15.4f} {sign}{delta:>9.4f}')
print('=' * 70)
print()
print('Interpretation:')
print(' - Pos-Neg gap: larger = embeddings cluster better by class')
print(' - TAR@FAR=1e-3 is the strictest verification threshold')
print(' - If fine-tuned beats stock, training improved domain-specific embeddings')

## Optional: DFW (Disguised Faces in the Wild) Evaluation

Only runs if `DFW_ROOT` exists. Computes TAR @ FAR=1e-4 per disguise type (helmets, sunglasses, masks).
Requires DFW pairs file or DFW folder structure with `<disguise_type>/<identity>/<image>.jpg`.

In [ ]:
# --- DFW pair loading + evaluation (skip if DFW not present) ---

def load_dfw_pairs(dfw_root):
    """Load DFW verification pairs grouped by disguise type."""
    pairs_file = os.path.join(dfw_root, 'pairs.txt')
    pairs_by_type = defaultdict(list)

    if not os.path.exists(pairs_file):
        print('No pairs.txt — building pairs from folder structure...')
        disguise_types = [d for d in os.listdir(dfw_root)
                          if os.path.isdir(os.path.join(dfw_root, d))]
        for dtype in disguise_types:
            dtype_dir = Path(dfw_root) / dtype
            identities = [d for d in dtype_dir.iterdir() if d.is_dir()]
            for ident in identities:
                imgs = sorted(f for f in ident.iterdir()
                              if f.suffix.lower() in ('.jpg', '.png'))
                for i in range(len(imgs)):
                    for j in range(i+1, min(i+3, len(imgs))):
                        pairs_by_type[dtype].append((str(imgs[i]), str(imgs[j]), 1))
            all_imgs_by_id = {ident.name: [f for f in ident.iterdir()
                                           if f.suffix.lower() in ('.jpg', '.png')]
                              for ident in identities}
            all_imgs_by_id = {k: v for k, v in all_imgs_by_id.items() if v}
            id_list = list(all_imgs_by_id.keys())
            if len(id_list) >= 2:
                for _ in range(min(500, len(id_list) * 2)):
                    id1, id2 = random.sample(id_list, 2)
                    pairs_by_type[dtype].append((
                        str(random.choice(all_imgs_by_id[id1])),
                        str(random.choice(all_imgs_by_id[id2])),
                        0,
                    ))
    else:
        with open(pairs_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 4:
                    dtype, img1, img2, label = parts[0], parts[1], parts[2], int(parts[3])
                    img1_path = os.path.join(dfw_root, img1)
                    img2_path = os.path.join(dfw_root, img2)
                    if os.path.exists(img1_path) and os.path.exists(img2_path):
                        pairs_by_type[dtype].append((img1_path, img2_path, label))
    return pairs_by_type


def evaluate_dfw(model, pairs_by_type):
    results = {}
    for dtype, pair_list in pairs_by_type.items():
        scores, labels = [], []
        for img1_path, img2_path, label in pair_list:
            e1 = get_embedding(model, img1_path)
            e2 = get_embedding(model, img2_path)
            scores.append(float(e1 @ e2))
            labels.append(label)
        scores = np.array(scores); labels = np.array(labels)
        fpr, tpr, _ = roc_curve(labels, scores)
        idx = np.searchsorted(fpr, 1e-4)
        tar = tpr[min(idx, len(tpr)-1)]
        results[dtype] = tar
        print(f'  {dtype}: TAR@FAR=1e-4 = {tar:.4f}')
    return results


if os.path.exists(DFW_ROOT):
    dfw_pairs = load_dfw_pairs(DFW_ROOT)
    for dtype, p in dfw_pairs.items():
        pos = sum(1 for _, _, l in p if l == 1)
        neg = sum(1 for _, _, l in p if l == 0)
        print(f'{dtype}: {len(p)} pairs ({pos} pos, {neg} neg)')

    print('\n=== Fine-tuned on DFW ===')
    ft_dfw = evaluate_dfw(model_ft, dfw_pairs)
    print('\n=== Stock on DFW ===')
    stock_dfw = evaluate_dfw(model_stock, dfw_pairs)

    print('\n' + '=' * 60)
    print(f'{"Disguise":<20} {"Stock":>10} {"Fine-tuned":>12} {"Delta":>8}')
    print('-' * 60)
    for dtype in sorted(set(list(ft_dfw.keys()) + list(stock_dfw.keys()))):
        s = stock_dfw.get(dtype, 0); f = ft_dfw.get(dtype, 0)
        print(f'{dtype:<20} {s:>10.4f} {f:>12.4f} {f-s:>+8.4f}')
    print('=' * 60)
else:
    print(f'DFW dataset not found at {DFW_ROOT} — skipping disguise eval.')
    print('To enable: download DFW dataset and place under that path.')